# using pydantic

In [1]:
import os
from langchain.chat_models import init_chat_model
os.environ['groq_api_key'] = os.getenv('groq_api_key')


In [2]:
model=init_chat_model("groq:qwen/qwen3-32b")
model

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000002950D5B4C20>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002950D5B5940>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [6]:
from pydantic import BaseModel,Field

In [4]:
!pip install pydantic


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
class movie(BaseModel):
    title:str = Field(description = 'the title of the movie')
    year:int = Field(description=  'this year the movie was released')
    director:str = Field(description='the director of the movie')
    rating:float=Field(description="The movies rating out of 10")

In [10]:
model_with_structure = model.with_structured_output(movie)

In [11]:
model_with_structure

RunnableBinding(bound=ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000002950D5B4C20>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002950D5B5940>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'the title of the movie', 'type': 'string'}, 'year': {'description': 'this year the movie was released', 'type': 'integer'}, 'director': {'description': 'the director of the movie', 'type': 'string'}, 'rating': {'description': 'The movies rating out of 10', 'type': 'number'}}, 'required': ['title', 'year', '

In [12]:
model.invoke('Provide details about the moview Inception')

AIMessage(content='<think>\nOkay, so I need to provide details about the movie Inception. Let me start by recalling what I know about it. First, the director is Christopher Nolan, right? And it stars Leonardo DiCaprio. The title "Inception" refers to the concept of planting an idea into someone\'s mind, which is a big part of the plot. The main character is Dom Cobb, a thief who steals information by infiltrating the subconscious. But in this movie, he\'s trying to do the opposite—plant an idea, which is called "inception."\n\nThe movie has a lot of layers, both literally and metaphorically. There are dreams within dreams, and each level has different time dilations. I remember something about a totem, which is used to determine if someone is in a dream or reality. Cobb\'s totem is a spinning top that wobbles when it stops, but it\'s supposed to be unique to him. He\'s afraid that he\'s stuck in a dream because of the trauma from his wife\'s death, which is a key point in the story.\n\

In [13]:
response=model_with_structure.invoke("Provide details about the moview avengers")
response

movie(title='Avengers', year=2012, director='Joss Whedon', rating=8.0)

# message output alongside parsed strructure 

In [14]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details."""
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)  

response = model_with_structure.invoke("Provide details about the movie Inception")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie Inception. I need to use the Movie function provided. Let me check the parameters required: title, year, director, and rating. I know the title is "Inception". The year it was released is 2010. The director is Christopher Nolan. As for the rating, it\'s popular, so maybe around 8.8 on IMDb. I should structure the JSON with these details. Let me make sure all required fields are included. Yep, that\'s all. Time to format the tool call correctly.\n', 'tool_calls': [{'id': 'pcgsjqwed', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 166, 'prompt_tokens': 231, 'total_tokens': 397, 'completion_time': 0.258530047, 'completion_tokens_details': {'reasoning_tokens': 118}, 'prompt_time': 0.012551454, 'prompt_tokens_detai

In [17]:
print(response)

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie Inception. I need to use the Movie function provided. Let me check the parameters required: title, year, director, and rating. I know the title is "Inception". The year it was released is 2010. The director is Christopher Nolan. As for the rating, it\'s popular, so maybe around 8.8 on IMDb. I should structure the JSON with these details. Let me make sure all required fields are included. Yep, that\'s all. Time to format the tool call correctly.\n', 'tool_calls': [{'id': 'pcgsjqwed', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 166, 'prompt_tokens': 231, 'total_tokens': 397, 'completion_time': 0.258530047, 'completion_tokens_details': {'reasoning_tokens': 118}, 'prompt_time': 0.012551454, 'prompt_tokens_detai

In [18]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Inception")
response

BadRequestError: Error code: 400 - {'error': {'message': "tool call validation failed: parameters for tool MovieDetails did not match schema: errors: [missing properties: 'cast', 'genres']", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<tool_call>\n{"name": "MovieDetails", "arguments": {"title": "Inception", "year": 2010}}\n</tool_call>'}}

In [19]:
from typing_extensions import TypedDict,Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]


model_withtypedict=model.with_structured_output(MovieDict)
response=model_withtypedict.invoke("Please provide the details of the movie avengers")
response


{'director': 'Joss Whedon', 'rating': 8, 'title': 'Avengers', 'year': 2012}

#  using dataclasses [i dont hav openai key  right now ..]
```
from pydantic import BaseModel, Field
from langchain.agents import create_agent


class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model="gpt-5",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result
```